# M1.4 · Logistic regression ve cross-entropy

Regresyondan sınıflandırmaya geçiş. Üç örnekte sigmoid, cross-entropy ve gradyanı elle hesaplıyor; `if` ile taşmayan bir sigmoid yazıyor, NumPy'da eğitiyor, PyTorch'ta logit'ten kararlı kaybı kendin kuruyor ve en sonda K sınıflı one-vs-rest modeli tek tensörle eğitiyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m1-4-logreg.html](https://m-gonul.github.io/pytorch-lab/m1-4-logreg.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m1_4_logreg.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def sig():\n        s = need(ns, \"sigmoid\")\n        close([s(-1), s(0), s(2)], [1 / (1 + math.e), 0.5, 1 / (1 + math.exp(-2))])\n        close([s(-1000), s(1000), s(-745.2)], [0.0, 1.0, 0.0], atol=1e-300)\n    R.test(\"sigmoid doğru ve ±1000'de taşmıyor\", sig)\n\n    def bc():\n        f = need(ns, \"bce\")\n        close([f(0.5, 1), f(0.25, 0)], [math.log(2), -math.log(0.75)])\n        ensure(math.isfinite(f(0.0, 1)) and math.isfinite(f(1.0, 0)), \"p = 0 ya da 1'de sonsuz kayıp; kırpma (eps) eksik\")\n    R.test(\"bce doğru ve p ∈ {0, 1}'de sonlu\", bc)\n\n    def gr():\n        out = need(ns, \"grad\")([[-1.0], [0.0], [2.0]], [0, 1, 1], [1.0], 0.0)\n        ensure(isinstance(out, (tuple, list)) and len(out) == 3, \"(gw, gb, L) döndürmeli\")\n        close(out[0], [-0.169116], atol=1e-5, what=\"gw\")\n        close(out[1], -0.116754, atol=1e-5, what=\"gb\")\n        close(out[2], 0.377779, atol=1e-5, what=\"L\")\n    R.test(\"elle örnek: gw ≈ −0.1691, gb ≈ −0.1168, L ≈ 0.3778\", gr)\n\n    def ft():\n        rng = random.Random(0)\n        X, y = [], []\n        for _ in range(60):\n            c = rng.random() < 0.5\n            X.append([rng.gauss(1.5 if c else -1.5, 0.8), rng.gauss(1.0 if c else -1.0, 0.8)])\n            y.append(int(c))\n        w, b = need(ns, \"fit\")(X, y, lr=0.5, steps=300)\n        acc = sum((sum(wi * xi for wi, xi in zip(w, x)) + b > 0) == bool(t) for x, t in zip(X, y)) / len(y)\n        ensure(acc >= 0.9, f\"eğitim doğruluğu {acc:.2f}, ≥ 0.90 bekleniyordu\")\n        return f\"2 boyutlu iki küme: doğruluk {acc:.2f}\"\n    R.test(\"fit iki kümeyi ayırıyor\", ft)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"sigmoid_np/predict içinde döngü yok\", lambda: (max_loops(src, 0, func=\"sigmoid_np\"), max_loops(src, 0, func=\"predict\")))\n    R.test(\"fit_logreg içinde yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"fit_logreg\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n\n    def sg():\n        z = np.array([-1000.0, -50, -1, 0, 2, 50, 1000])\n        with np.errstate(over=\"raise\", invalid=\"raise\", divide=\"raise\", under=\"ignore\"):\n            s = need(ns, \"sigmoid_np\")(z)\n        want = np.array([0.0, 1.9287498479639178e-22, 0.2689414213699951, 0.5, 0.8807970779778823, 1.0, 1.0])\n        close(s, want, rtol=1e-9, atol=1e-30)\n        return \"±1000'de bile uyarı/taşma yok\"\n    R.test(\"sigmoid_np kararlı (taşma/geçersiz değer hatası yok)\", sg)\n\n    rng = np.random.default_rng(5)\n    X = np.concatenate([rng.normal(-1.2, 1, (120, 3)), rng.normal(1.2, 1, (120, 3))])\n    y = np.repeat([0, 1], 120)\n\n    def one():\n        w, b, L = need(ns, \"fit_logreg\")(np.array([[-1.0], [0.0], [2.0]]), np.array([0, 1, 1]), lr=1.0, steps=1)\n        close(np.asarray(L)[0], 0.6931471805599453, what=\"ilk kayıp (w = b = 0)\")\n        close([float(np.ravel(w)[0]), b], [-1.0 * (0.5 * -1 + -0.5 * 0 + -0.5 * 2) / 3, -1.0 * (0.5 - 0.5 - 0.5) / 3], what=\"tek adım sonrası (w, b)\")\n    R.test(\"tek adım: sıfırdan başlayıp doğru gradyanla güncelliyor\", one)\n\n    def tr():\n        w, b, L = need(ns, \"fit_logreg\")(X, y, lr=0.5, steps=500)\n        L = np.asarray(L)\n        ensure(L.shape == (500,), f\"losses shape {L.shape}\")\n        ensure(L[-1] < L[0], \"kayıp azalmadı\")\n        acc = (np.asarray(need(ns, \"predict\")(X, w, b)) == y).mean()\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"doğruluk {acc:.3f}, kayıp {L[0]:.3f} → {L[-1]:.3f}\"\n    R.test(\"eğitim + predict\", tr)\n",
 "d": "import ast\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def no_builtin():\n        t = ast.parse(src)\n        for node in ast.walk(t):\n            if isinstance(node, ast.FunctionDef) and node.name == \"bce_logits\":\n                names = {n.attr for n in ast.walk(node) if isinstance(n, ast.Attribute)} | {n.id for n in ast.walk(node) if isinstance(n, ast.Name)}\n                ensure(not ({\"binary_cross_entropy_with_logits\", \"binary_cross_entropy\", \"BCEWithLogitsLoss\", \"BCELoss\"} & names), \"bce_logits'i hazır kayıp fonksiyonuyla yazma; formülü kendin uygula\")\n                return\n        raise AssertionError(\"`bce_logits` bulunamadı\")\n    R.test(\"bce_logits hazır kayıp fonksiyonu kullanmıyor\", no_builtin)\n\n    z = torch.tensor([-1.0, 0.0, 2.0, -100.0, 100.0, 30.0], dtype=torch.float64)\n    y = torch.tensor([0.0, 1.0, 1.0, 1.0, 0.0, 1.0], dtype=torch.float64)\n\n    def val():\n        out = need(ns, \"bce_logits\")(z, y)\n        ensure(torch.isfinite(out), f\"sonuç sonlu değil: {out}\")\n        close(out, F.binary_cross_entropy_with_logits(z, y))\n        close(need(ns, \"bce_logits\")(z[:3], y[:3]), 0.3777789, atol=1e-6, what=\"elle örnek\")\n    R.test(\"bce_logits PyTorch'unkiyle eşleşiyor (z = ±100 dahil)\", val)\n\n    def gr():\n        zz = z.clone().requires_grad_(True)\n        need(ns, \"bce_logits\")(zz, y).backward()\n        close(zz.grad, (torch.sigmoid(z) - y) / len(z), what=\"∂L/∂z\")\n        return \"∂L/∂z = (σ(z) − y)/n, z = 0 dahil\"\n    R.test(\"autograd gradyanı (σ(z) − y)/n — z = 0 noktasında da\", gr)\n\n    def tr():\n        g = torch.Generator().manual_seed(1)\n        X = torch.cat([torch.randn(100, 2, generator=g) - 1.3, torch.randn(100, 2, generator=g) + 1.3])\n        t = torch.cat([torch.zeros(100), torch.ones(100)])\n        torch.manual_seed(0)\n        m = need(ns, \"train_torch\")(X, t, lr=0.5, steps=400)\n        ensure(isinstance(m, nn.Linear), \"nn.Linear döndürmeli\")\n        with torch.no_grad():\n            acc = ((m(X).squeeze(1) > 0).float() == t).float().mean().item()\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"doğruluk {acc:.3f}\"\n    R.test(\"train_torch iki kümeyi ayırıyor\", tr)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef binary(X, t, lr, steps):\n    w = torch.zeros(X.shape[1]); b = torch.zeros(())\n    for _ in range(steps):\n        r = torch.sigmoid(X @ w + b) - t\n        w = w - lr * X.T @ r / len(t)\n        b = b - lr * r.mean()\n    return w, b\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"train_ovr içinde yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"train_ovr\", why=\"sınıflar üzerinde döngü olmamalı\"))\n    R.test(\"predict_ovr içinde döngü yok\", lambda: max_loops(src, 0, func=\"predict_ovr\"))\n    g = torch.Generator().manual_seed(3)\n    centers = torch.tensor([[0.0, 3.0], [-3.0, -2.0], [3.0, -2.0], [0.0, -4.5]])\n    y = torch.arange(4).repeat_interleave(60)\n    X = centers[y] + torch.randn(240, 2, generator=g)\n\n    def eq():\n        W, b = need(ns, \"train_ovr\")(X, y, 4, lr=0.5, steps=80)\n        ensure(tuple(W.shape) == (2, 4) and tuple(b.shape) == (4,), f\"shape'ler {tuple(W.shape)}, {tuple(b.shape)}\")\n        for k in range(4):\n            wk, bk = binary(X, (y == k).float(), 0.5, 80)\n            close(W[:, k], wk, rtol=1e-4, atol=1e-5, what=f\"sınıf {k} ağırlıkları\")\n            close(b[k], bk, rtol=1e-4, atol=1e-5, what=f\"sınıf {k} bias\")\n        return \"4 sütunun her biri ayrı eğitilmiş ikili modelle aynı\"\n    R.test(\"K ayrı ikili modelle eşleşiyor\", eq)\n\n    def acc():\n        W, b = need(ns, \"train_ovr\")(X, y, 4)\n        a = (need(ns, \"predict_ovr\")(X, W, b) == y).float().mean().item()\n        ensure(a >= 0.85, f\"doğruluk {a:.2f}\")\n        return f\"4 sınıf, doğruluk {a:.3f}\"\n    R.test(\"predict_ovr doğruluğu ≥ 0.85\", acc)\n    if \"train_ovr\" not in ns:\n        return\n    K = 10\n    Xb = torch.randn(2000, 20, generator=g)\n    yb = torch.randint(0, K, (2000,), generator=g)\n    t_loop = bench(lambda: [binary(Xb, (yb == k).float(), 0.5, 100) for k in range(K)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"train_ovr\"](Xb, yb, K, lr=0.5, steps=100), repeat=3, number=1)\n    R.table(f\"{K} sınıf × 100 adım (n = 2000, d = 20)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for k: K ayrı ikili model\", fmt_time(t_loop), \"1×\"],\n             [\"senin train_ovr\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]],\n            note=\"Kazancın kaynağı: K adet (n×d)·(d,) çarpımı yerine tek (n×d)·(d×K) matris çarpımı.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m1_4_logreg as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Lineer modelin çıktısı $z=w^\top x+b$ herhangi bir gerçek sayı; ona logit denir. Sınıflandırmada bir olasılık istiyoruz, bu yüzden $z$'yi sigmoid'den geçiriyoruz:

$$
p=\sigma(z)=\frac{1}{1+e^{-z}},\qquad \ell(p,y)=-\big[y\ln p+(1-y)\ln(1-p)\big].
$$

$\ell$, ikili cross-entropy: doğru sınıfa verilen olasılığın negatif logaritması. Zincir kuralı sürpriz derecede sade bir gradyan verir:

$$
\frac{\partial \ell}{\partial z}=p-y,\qquad \nabla_w L=\frac1n X^\top(p-y),\qquad \frac{\partial L}{\partial b}=\frac1n\sum_i(p_i-y_i).
$$

Bu, linear regression'daki $\frac2n X^\top e$ ile aynı biçimde: "tahmin eksi hedef, girdilerle ağırlıklandırılmış". Fark yalnız $e$'nin artık bir olasılık farkı olması.

Sayısal tuzak: $z=-1000$ için `exp(-z)` taşar. Kararlı yazım $z$'nin işaretine göre iki eşdeğer formdan birini seçer; bu modülde `if`'in gerçek bir işi var.

## 1 · Elle

Hesap makinesi yerine şu değerleri kullan:

| üstel | logaritma |
|---|---|
| $e^{1}\approx 2.7183$ | $\ln 0.5\approx-0.6931$ |
| $e^{-2}\approx0.1353$ | $\ln 0.7311\approx-0.3133$ |
| | $\ln 0.8808\approx-0.1269$ |

<!-- çözüm -->

Önce üç olasılığı, sonra üç kaybı hesapla. $\ell_2$ en büyük kayıp: model $x=0$ noktasında tam kararsız ($p=0.5$) ve doğru cevap 1. $\ell_3$ en küçük: model emin ve haklı.

Gradyanın işaretlerini yorumla: $\partial L/\partial w<0$, yani $w$'yi **artırmak** kaybı azaltır (eğimi dikleştir, sınıfları daha keskin ayır).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Veri $x=[-1,\,0,\,2]$, $y=[0,\,1,\,1]$; $w=1$, $b=0$, yani $z=x$. Olasılıklar $p=\sigma(z)$**

- `p1`: $p_1=\sigma(-1)$
- `p2`: $p_2=\sigma(0)$
- `p3`: $p_3=\sigma(2)$

**Örnek başına kayıp $\ell_i=-[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$ ve ortalama**

- `l1`: $\ell_1$ ($y_1=0$)
- `l2`: $\ell_2$ ($y_2=1$)
- `l3`: $\ell_3$ ($y_3=1$)
- `L`: $L=\frac13\sum\ell_i$

**Gradyan: $\partial\ell_i/\partial z_i=p_i-y_i$**

- `gw`: $\partial L/\partial w=\frac13\sum(p_i-y_i)x_i$
- `gb`: $\partial L/\partial b=\frac13\sum(p_i-y_i)$
- `acc`: Doğruluk ($p\ge0.5\Rightarrow1$), 0–1 arası

In [ ]:
cevaplar = dict(
    p1=None,
    p2=None,
    p3=None,
    l1=None,
    l2=None,
    l3=None,
    L=None,
    gw=None,
    gb=None,
    acc=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"p1":[0.268941,0.002],"p2":[0.5,0.001],"p3":[0.880797,0.002],"l1":[0.313262,0.002],"l2":[0.693147,0.002],"l3":[0.126928,0.002],"L":[0.377779,0.002],"gw":[-0.169116,0.002],"gb":[-0.116754,0.002],"acc":[1,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Listelerle yaz. `X` satırları örnek olan iç içe liste (`n × d`), `w` uzunluğu `d` olan liste.

- `sigmoid(z)` — **taşmamalı**: $z\ge0$ için $1/(1+e^{-z})$, $z<0$ için $e^{z}/(1+e^{z})$. `sigmoid(-1000)` hata vermeden `0.0` döndürmeli.
- `bce(p, y, eps=1e-12)` — tek örnek kaybı; $p$'yi `[eps, 1-eps]` aralığına kırp.
- `grad(X, y, w, b)` → `(gw, gb, L)`; `gw` liste.
- `fit(X, y, lr=0.5, steps=1000)` → `(w, b)`.

`numpy`/`torch` yasak; `math.exp`, `math.log` serbest.

In [ ]:
%%writefile stage_b.py
import math


def sigmoid(z):
    ...


def bce(p, y, eps=1e-12):
    ...


def grad(X, y, w, b):
    """→ (gw: liste, gb, ortalama kayıp)"""
    ...


def fit(X, y, lr=0.5, steps=1000):
    ...


X = [[-1.0], [0.0], [2.0]]
y = [0, 1, 1]
print(grad(X, y, [1.0], 0.0))      # ([-0.169…], -0.1167…, 0.3777…)
print(sigmoid(-1000), sigmoid(1000))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da kararlı sigmoid için `if` yerine iki dalı birden vektörel hesaplayıp seçeriz. Ama dikkat: `np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))` **her iki** dalı da tüm elemanlar için hesaplar ve büyük $|z|$'de taşma uyarısı verir. Taşmayan hile: $e^{-|z|}$ her zaman $\le1$.

$$
q=e^{-|z|},\qquad \sigma(z)=\begin{cases}1/(1+q) & z\ge0\\ q/(1+q) & z<0\end{cases}
$$

- `sigmoid_np(z)` — taşma uyarısı **yok** (kontrol, `np.errstate(over="raise", invalid="raise")` altında çağırır; $e^{-1000}$'in 0'a yuvarlanması — underflow — zararsızdır ve serbest).
- `fit_logreg(X, y, lr=0.5, steps=500)` → `(w, b, losses)`; yalnız adım döngüsü.
- `predict(X, w, b)` → 0/1 tamsayı dizisi.

In [ ]:
%%writefile stage_c.py
import numpy as np


def sigmoid_np(z):
    ...


def fit_logreg(X, y, lr=0.5, steps=500):
    """→ (w, b, losses). Yalnız adım döngüsü."""
    ...


def predict(X, w, b):
    ...


rng = np.random.default_rng(0)
X = np.concatenate([rng.normal(-1.5, 1, (100, 2)), rng.normal(1.5, 1, (100, 2))])
y = np.repeat([0, 1], 100)
w, b, losses = fit_logreg(X, y)
print("doğruluk:", (predict(X, w, b) == y).mean(), "son kayıp:", losses[-1])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Kaybı olasılıktan değil doğrudan logit'ten hesaplamak hem daha kararlı hem daha hızlı. $\ln\sigma(z)=-\ln(1+e^{-z})$ ve $\ln(1-\sigma(z))=-z-\ln(1+e^{-z})$ yerine konunca:

$$
\ell(z,y)=\operatorname{softplus}(z)-zy,\qquad \operatorname{softplus}(z)=\ln(1+e^{z}).
$$

$\operatorname{softplus}$'ı taşmadan hesaplamanın iki yolu var ve **ikisi aynı sayıyı verir**:

```python
torch.logaddexp(torch.zeros_like(z), z)            # ln(e⁰ + eᶻ)
z.clamp(min=0) + torch.log1p(torch.exp(-z.abs()))  # max(z,0) + ln(1+e^{-|z|})
```

Ama ikincisinin **gradyanı** $z=0$'da yanlış: `clamp` ve `abs` iki ayrı kırılma noktası; autograd her birinde bir alt-gradyan seçer ve seçimler birbirini tutmaz. Gerçek fonksiyon pürüzsüz ve türevi $\sigma(0)=0.5$, autograd ise 0 ya da 1 bulur. Kontrol $z=0$'ı test eder.

- `bce_logits(z, y)` — ortalama kayıp. Hazır BCE fonksiyonlarını kullanma; kontrol seninkini `F.binary_cross_entropy_with_logits` ile karşılaştıracak, $z=\pm100$ ve $z=0$ dahil.
- `train_torch(X, y, lr=0.5, steps=500)` → eğitilmiş `nn.Linear(d, 1)`; `nn.BCEWithLogitsLoss` ve `optim.SGD` ile.

Kontrol ayrıca autograd'ın `bce_logits` üzerinden bulduğu $\partial L/\partial z$'nin $(\sigma(z)-y)/n$ olduğunu doğrular.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


def bce_logits(z, y):
    """Kararlı ikili cross-entropy (ortalama), doğrudan logit'ten."""
    ...


def train_torch(X, y, lr=0.5, steps=500):
    """→ eğitilmiş nn.Linear(d, 1)"""
    ...


z = torch.tensor([-1.0, 0.0, 2.0, -100.0, 100.0])
y = torch.tensor([0.0, 1.0, 1.0, 1.0, 0.0])
print(bce_logits(z, y))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

$K$ sınıf için en basit yaklaşım **one-vs-rest**: her sınıf için "bu sınıf mı, değil mi?" diye ayrı bir ikili sınıflandırıcı; tahmin, en yüksek logit'i veren sınıf. $K$ modeli döngüyle eğitmek yerine:

$$
Z = XW + b \in\mathbb{R}^{n\times K},\qquad Y_{ik}=[y_i=k]\ \text{(one-hot)},\qquad
\nabla_W L=\frac1n X^\top\big(\sigma(Z)-Y\big).
$$

- `train_ovr(X, y, K, lr=0.5, steps=300)` → `(W: (d, K), b: (K,))`. Kayıp: her sınıfın ortalama BCE'sinin **toplamı**; başlangıç sıfır; tam-batch gradient descent. Yalnız adım döngüsü.
- `predict_ovr(X, W, b)` → `(n,)` sınıf indeksleri.

Kontrol, sonucu K ayrı ikili modelle karşılaştırır ve hız tablosu üretir.

In [ ]:
%%writefile stage_e.py
import torch


def train_ovr(X, y, K, lr=0.5, steps=300):
    """→ (W: (d, K), b: (K,)). Yalnız adım döngüsü; sınıflar üzerinde döngü yok."""
    n, d = X.shape
    Y = torch.nn.functional.one_hot(y, K).float()   # (n, K)
    W = torch.zeros(d, K)
    b = torch.zeros(K)
    ...


def predict_ovr(X, W, b):
    ...


torch.manual_seed(0)
centers = torch.tensor([[0.0, 3.0], [-3.0, -2.0], [3.0, -2.0]])
y = torch.arange(3).repeat_interleave(100)
X = centers[y] + torch.randn(300, 2)
W, b = train_ovr(X, y, 3)
print("doğruluk:", (predict_ovr(X, W, b) == y).float().mean().item())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Lin, Weng, Keerthi (2008): lojistik regresyon için güven bölgesi Newton

[Lin, Weng, Keerthi — Trust Region Newton Method for Large-Scale Logistic Regression (JMLR 2008)](https://www.jmlr.org/papers/v9/lin08b.html)

L2 düzenlemeli lojistik regresyonu (a) gradyan inişi, (b) tam Newton / IRLS, (c) makalenin güven bölgesli Newton'u (iç döngüde eşlenik gradyanla Hessian-vektör çarpımı, Hessian'ı hiç oluşturmadan) ile çöz. Yüksek boyutlu seyrek bir veri setinde (ör. 20 Newsgroups'un TF-IDF'i) yinelenme ve süre karşılaştırması yap; Hessian-vektör çarpımının $X^\top(D(Xv))$ biçimiyle O(nnz) olduğunu göster.


**Adımlar**

1. Eşlenik gradyan iç döngüsünü güven bölgesi sınırında kes (Steihaug).

**Kabul ölçütleri**

- TRON, yüksek boyutta GD ve tam Newton'dan hızlı yakınsar.
- Hessian'sız Hv doğruluğu sonlu farkla doğrulanmış.

## Kaynaklar

- [Stanford CS229 — Lecture notes: Classification and logistic regression](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [CS231n — Linear classification](https://cs231n.github.io/linear-classify/) — MIT
- [Dive into Deep Learning §4 Linear Neural Networks for Classification](https://d2l.ai/chapter_linear-classification/index.html) — CC BY-SA 4.0 / MIT-0
- [PyTorch — BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) — BSD-3